CPT модели

In [ ]:
import ctypes
import unsloth
from unsloth import FastLanguageModel
from transformers import TrainingArguments, AutoTokenizer
from trl import SFTTrainer
import torch
import torchvision
import json, random, gc, os
from datasets import Dataset
import numpy as np
from tqdm import tqdm

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 512
OUTPUT_DIR = "./qwen_cpt_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CORPUS_PATH = "./cpt_corpus.jsonl"  #файл c подготовленным CPT датасетом

1. ЗАГРУЗКА БАЗОВОЙ МОДЕЛИ

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
tokenizer.pad_token = tokenizer.eos_token

2. ЗАГРУЗКА CPT-КОРПУСА

In [ ]:
def load_cpt_corpus(path, max_samples=None):
    texts = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            item = json.loads(line)
            text = item.get("text", "").strip()
            if text and len(text) >= 100:
                texts.append(text)
    print(f"Загружено {len(texts)} текстов из корпуса")
    return texts

In [ ]:
corpus_texts = load_cpt_corpus(CORPUS_PATH)

3. ПОДГОТОВКА ДАТАСЕТА ДЛЯ CPT

In [ ]:
def prepare_cpt_dataset(texts, tokenizer):
    formatted = []
    for text in texts:
        #токенизация и приведение к длине MAX_SEQ_LENGTH
        tokens = tokenizer(
            text,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            padding=False,
            return_tensors=None,  #возвращаем список, не тензор
        )
        #декодирование обратно в текст
        truncated_text = tokenizer.decode(
            tokens["input_ids"],
            skip_special_tokens=True
        )
        if len(tokens["input_ids"]) >= 50:  #фильтр слишком коротких
            formatted.append({"text": truncated_text})
    print(f"Итого примеров после токенизации: {len(formatted)}")
    return Dataset.from_list(formatted)

In [ ]:
cpt_dataset = prepare_cpt_dataset(corpus_texts, tokenizer)

4. НАСТРОЙКА LoRA ДЛЯ CPT

In [ ]:
FastLanguageModel.for_training(model)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=RANDOM_SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

5. АРГУМЕНТЫ ОБУЧЕНИЯ (для CPT: lr меньше, эпоха одна, батч маленький)

In [ ]:
training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "cpt_checkpoints"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=5e-5,
    fp16=False,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to="none",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    torch_compile=False,
    dataloader_pin_memory=False,
)

6. ОБУЧЕНИЕ

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=cpt_dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    dataset_num_proc=1,
    packing=False,
)

In [ ]:
trainer.train()

In [ ]:
cpt_adapter_path = "./qwen_cpt_results/cpt_lora_adapter"

model.save_pretrained(cpt_adapter_path, save_adapter=True)
tokenizer.save_pretrained(cpt_adapter_path)

7. СОХРАНЕНИЕ CPT LoRA-АДАПТЕРА

In [ ]:
cpt_adapter_path = os.path.join(OUTPUT_DIR, "cpt_lora_adapter")

In [ ]:
model.save_pretrained(cpt_adapter_path, save_adapter=True)
tokenizer.save_pretrained(cpt_adapter_path)

Оценка забывания (сравнение базовой модели и модели после CPT)

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
CPT_ADAPTER_PATH = "./qwen_cpt_results/cpt_lora_adapter"
MAX_SEQ_LENGTH = 512

Загрузка базовой модели

In [ ]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

Загрузка CPT модели

In [ ]:
#загрузка CPT модели с адаптером
cpt_model, cpt_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    adapter_name=CPT_ADAPTER_PATH,
)
cpt_model.eval()

Тестовые задания

Базовые знания

In [ ]:
WORLD_KNOWLEDGE_TESTS = [
    {"question": "What is the capital of France?", "expected": "Paris", "type": "fact"},
    {"question": "Who wrote 'Romeo and Juliet'?", "expected": "Shakespeare", "type": "fact"},
    {"question": "What is the chemical symbol for gold?", "expected": "Au", "type": "fact"},
    {"question": "What is the largest planet in our solar system?", "expected": "Jupiter", "type": "fact"},
    {"question": "Who painted the Mona Lisa?", "expected": "da Vinci", "type": "fact"},
    {"question": "What is the square root of 144?", "expected": "12", "type": "math"},
    {"question": "What year did World War II end?", "expected": "1945", "type": "fact"},
    {"question": "What is the hardest natural substance?", "expected": "diamond", "type": "fact"},
]

Логическое рассуждение

In [ ]:
REASONING_TESTS = [
    {"question": "If all birds can fly and penguins are birds, can penguins fly? Answer yes or no.", "expected": "no", "type": "logic"},
    {"question": "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Answer with a number.", "expected": "0.05", "type": "math"},
    {"question": "If you have me, you want to share me. If you share me, you no longer have me. What am I? Answer with one word.", "expected": "secret", "type": "logic"},
    {"question": "Which is heavier: 1 kg of feathers or 1 kg of steel? Answer with 'feathers', 'steel', or 'same'.", "expected": "same", "type": "logic"},
]

Commonsense reasoning

In [ ]:
COMMONSENSE_TESTS = [
    {"question": "You wake up in the morning, what do you do first? A) Go to bed B) Brush teeth C) Open eyes D) Eat dinner", "expected": "C", "type": "commonsense"},
    {"question": "If you drop a glass on concrete floor, it will most likely: A) Bounce B) Float C) Break D) Disappear", "expected": "C", "type": "commonsense"},
    {"question": "To stay dry in the rain, you should use: A) Umbrella B) Towel C) Hair dryer D) Fan", "expected": "A", "type": "commonsense"},
]

Языковое понимание

In [ ]:
LANGUAGE_TESTS = [
    {"question": "The word 'happy' is opposite of: A) Sad B) Joyful C) Glad D) Cheerful", "expected": "A", "type": "language"},
    {"question": "Which word is a synonym for 'quick'? A) Slow B) Fast C) Lazy D) Heavy", "expected": "B", "type": "language"},
    {"question": "Complete the sentence: 'She ___ to the store yesterday.' A) go B) goes C) went D) going", "expected": "C", "type": "language"},
]

Доменная специфика

In [ ]:
DOMAIN_TESTS = [
    {"question": "A GitHub issue describes a NullPointerException when saving user profile. Is this an anomaly or OK?", "expected": "Anomaly", "type": "domain"},
    {"question": "A user asks for a new API endpoint to get user stats. Is this an anomaly or OK?", "expected": "OK", "type": "domain"},
    {"question": "Code review: Function returns wrong value for negative input. Should this be marked as anomaly or OK?", "expected": "Anomaly", "type": "domain"},
]

Функция для генерации ответа

In [ ]:
def generate_answer(model, tokenizer, question, max_new_tokens=50):
    messages = [
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

Функция ниже проверяет, содержит ли ответ ожидаемое значение

In [ ]:
def evaluate_answer(response, expected):
    response_lower = response.lower()
    expected_lower = expected.lower()

    if expected_lower in response_lower:
        return True

    if len(expected_lower) == 1 and expected_lower in "abcd":
        if f"{expected_lower})" in response_lower or f"{expected_lower}." in response_lower:
            return True
    return False

Оценка моделей

In [ ]:
def evaluate_model(model, tokenizer, tests, model_name):

    results = {"fact": 0, "math": 0, "logic": 0, "commonsense": 0, "language": 0, "domain": 0}
    total = {"fact": 0, "math": 0, "logic": 0, "commonsense": 0, "language": 0, "domain": 0}

    for test in tests:
        q_type = test["type"]
        total[q_type] += 1

        response = generate_answer(model, tokenizer, test["question"], max_new_tokens=50)
        correct = evaluate_answer(response, test["expected"])

        if correct:
            results[q_type] += 1

        #вывод нескольких примеров
        if len([t for t in tests if t["question"] == test["question"]]) < 3:
            status = "OK" if correct else "Incorrect"
            print(f"\n{status} {test['question'][:80]}...")
            print(f"Ответ: {response[:100]}")
            print(f"Ожидалось: {test['expected']}")

    #итоги по категориям
    print(f"\n ИТОГИ {model_name}")
    total_correct = sum(results.values())
    total_questions = sum(total.values())
    print(f"Общий accuracy: {total_correct}/{total_questions} = {100*total_correct/total_questions:.1f}%")

    for cat in results:
        if total[cat] > 0:
            acc = results[cat] / total[cat]
            print(f"{cat:12s}: {results[cat]:2d}/{total[cat]} ({100*acc:.1f}%)")

    return results, total

Запуск оценки

In [ ]:
all_tests = WORLD_KNOWLEDGE_TESTS + REASONING_TESTS + COMMONSENSE_TESTS + LANGUAGE_TESTS + DOMAIN_TESTS

#оценка базовой модели
base_results, base_total = evaluate_model(base_model, base_tokenizer, all_tests, "BASE MODEL")
#оценка CPT модели
cpt_results, cpt_total = evaluate_model(cpt_model, cpt_tokenizer, all_tests, "CPT MODEL")

Сравнение

In [ ]:
categories = ["fact", "math", "logic", "commonsense", "language", "domain"]
for cat in categories:
    if base_total[cat] > 0:
        base_acc = base_results[cat] / base_total[cat] if base_total[cat] > 0 else 0
        cpt_acc = cpt_results[cat] / cpt_total[cat] if cpt_total[cat] > 0 else 0
        diff = cpt_acc - base_acc

        if diff >= -0.05:
            status = "OK"
        elif diff >= -0.15:
            status = "Слабое забывание"
        else:
            status = "Критическое забывание"

        print(f"\n{cat.upper():12s}: Base={base_acc:.1%} → CPT={cpt_acc:.1%} (Δ={diff:+.1%}) {status}")

total_base = sum(base_results.values()) / sum(base_total.values())
total_cpt = sum(cpt_results.values()) / sum(cpt_total.values())
total_diff = total_cpt - total_base

print(f"\n{'ИТОГО':12s}: Base={total_base:.1%} → CPT={total_cpt:.1%} (Δ={total_diff:+.1%})")

if total_diff >= -0.03:
    print("\nЗабывания нет или оно минимально")
elif total_diff >= -0.10:
    print("\nЕсть небольшое забывание")
else:
    print("\nСильное катастрофическое забывание")